In [ ]:
from pathlib import Path
import copy
import numpy as np
import pandas as pd
from tqdm import tqdm
from tqdm.auto import tqdm
from sklearn.preprocessing import OneHotEncoder
from collections import Counter
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
import voyageai
from getpass import getpass
import os
import json
import pickle

PROJECT_ROOT = Path.cwd().parent

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
INTERIM_DIR = PROJECT_ROOT / 'data' / 'interim'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'

INTERIM_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
(ARTIFACTS_DIR / 'mappings').mkdir(parents=True, exist_ok=True)

print(f'프로젝트 루트 | Project root: {PROJECT_ROOT}')
print(f'원본 데이터 경로 | Raw data: {RAW_DIR}')
print(f'처리된 데이터 경로 | Processed data: {PROCESSED_DIR}')

In [ ]:
products = pd.read_csv(RAW_DIR / 'products_train.csv')
sessions = pd.read_csv(RAW_DIR / 'sessions_train.csv')

print(
    f'상품 데이터 크기 | Products shape: {products.shape}'
)

print(
    f'세션 데이터 크기 | Sessions shape: {sessions.shape}'
)

In [ ]:
products.head()

In [ ]:
sessions.head()

In [ ]:
def normalize_empty_string(df): 
    print(f"결측 행 처리 | NULL ROW PROCESSING")
    print(f"---" * 30)
    df_cleaned_strings = copy.deepcopy(df)
    
    for column in df_cleaned_strings.select_dtypes(include = ['object']).columns: 
        mask = (
            df_cleaned_strings[column]
            .astype('string')
            .str.strip()
            .eq('')
        )
        
        df_cleaned_strings.loc[mask, column] = pd.NA
    
    empty_values_before = 0
        
    for column in df.columns: 
        print(f"정제 전 |{column}| 컬럼의 결측치 수 | Number of missing values in column |{column}| before cleaning : {df[column].astype('string').str.strip().eq('').sum()}")
        empty_values_before += df[column].astype('string').str.strip().eq('').sum()
    
    print(f"정제 전 총 결측치 수 | Total missing values before cleaning : {empty_values_before}")
    
    
    print(f'---' * 30)
    
    empty_values_after = 0 
    
    for column in df_cleaned_strings.columns:
        print(f"정제 전 |{column}| 컬럼의 결측치 수 | Number of missing values in column |{column}| before cleaning : {df_cleaned_strings[column].eq('').sum()}")
        empty_values_after += df_cleaned_strings[column].eq('').sum()
    
    print(f"정제 후 총 결측치 수 | Total missing values after cleaning : {empty_values_after}")
    
    if empty_values_after == 0:
        print(f'---' * 30) 
        print(f'정제 성공! | Cleaning successful!')
        return df_cleaned_strings
    else:
        print(f'---' * 30)
        print(f"파이프라인 오류! | Pipeline error!")
    
     
        

In [ ]:
products1 = normalize_empty_string(products)
products1.head()


In [ ]:
sessions1 = normalize_empty_string(sessions)
sessions1.head()

In [ ]:
def parse_sessions(sessions, products):
    print(
        '세션 처리 및 임베딩 레이블링 생성 | '
        'SESSION PROCESSING AND EMBEDDING MARKUP CREATION'
    )
    print('---' * 25)

    catalog = products.copy()
    sessions_hist = sessions.copy()

    catalog.columns = (
        catalog.columns
        .astype(str)
        .str.strip()
        .str.lower()
    )

    sessions_hist.columns = (
        sessions_hist.columns
        .astype(str)
        .str.strip()
        .str.lower()
    )

    if catalog.columns.duplicated().any():
        raise ValueError(
            '상품 데이터에 중복 컬럼명이 있습니다 | '
            'Duplicate product columns after normalization'
        )

    if sessions_hist.columns.duplicated().any():
        raise ValueError(
            '세션 데이터에 중복 컬럼명이 있습니다 | '
            'Duplicate session columns after normalization'
        )

    required_product_columns = {'id'}
    required_session_columns = {'prev_items', 'next_item', 'locale'}

    missing_product_columns = (
        required_product_columns - set(catalog.columns)
    )

    missing_session_columns = (
        required_session_columns - set(sessions_hist.columns)
    )

    if missing_product_columns:
        raise KeyError(
            f'상품 데이터 필수 컬럼 누락 | '
            f'Missing required product columns: '
            f'{missing_product_columns}'
        )

    if missing_session_columns:
        raise KeyError(
            f'세션 데이터 필수 컬럼 누락 | '
            f'Missing required session columns: '
            f'{missing_session_columns}'
        )

    if catalog['id'].isna().any():
        raise ValueError(
            '카탈로그 ID에 결측치가 있습니다 | '
            'Missing catalog IDs detected'
        )

    if sessions_hist[
        ['prev_items', 'next_item', 'locale']
    ].isna().any().any():
        raise ValueError(
            '세션 핵심 컬럼에 결측치가 있습니다 | '
            'Missing values detected in core session columns'
        )

    with tqdm(total=6, desc='Session Pipeline') as pbar:

        pbar.set_description('1/6 Building catalog universe')

        catalog_asins = pd.Index(
            catalog['id'].drop_duplicates()
        )

        repeated_asin_rows = (
            catalog['id'].duplicated().sum()
        )

        print(
            f'카탈로그 전체 행 수 | '
            f'Total catalog rows : {len(catalog):,}'
        )

        print(
            f'고유 ASIN 수 | '
            f'Unique ASINs : {len(catalog_asins):,}'
        )

        print(
            f'다른 지역에서 반복되는 ASIN 행 수 | '
            f'Repeated ASIN rows across locales : '
            f'{repeated_asin_rows:,}'
        )

        print('---' * 25)

        pbar.update(1)

        pbar.set_description('2/6 Parsing session histories')

        parsed_series = (
            sessions_hist['prev_items']
            .astype('string')
            .str.findall(r"'([^']+)'")
        )

        empty_sessions = (
            parsed_series
            .str.len()
            .eq(0)
            .sum()
        )

        if empty_sessions > 0:
            raise ValueError(
                f'상품이 없는 세션 발견 | '
                f'Empty parsed sessions detected: '
                f'{empty_sessions:,}'
            )

        parsed_lists = parsed_series.tolist()

        del parsed_series

        print(
            f'파싱된 세션 수 | '
            f'Parsed sessions : {len(parsed_lists):,}'
        )

        print('---' * 25)

        pbar.update(1)

        pbar.set_description('3/6 Building behavioral universe')

        behavioral_set = set(
            sessions_hist['next_item'].unique()
        )

        for row in tqdm(
            parsed_lists,
            desc='Collecting unique history ASINs',
            leave=False
        ):
            behavioral_set.update(row)

        behavioral_index = pd.Index(
            list(behavioral_set)
        )

        missing_behavioral = behavioral_index[
            ~behavioral_index.isin(catalog_asins)
        ]

        if len(missing_behavioral) > 0:
            raise ValueError(
                f'카탈로그에 없는 행동 상품 발견 | '
                f'Behavioral items missing from catalog: '
                f'{len(missing_behavioral):,}'
            )

        behavioral_mask = catalog_asins.isin(
            behavioral_set
        )

        behavioral_asins = catalog_asins[
            behavioral_mask
        ]

        cold_asins = catalog_asins[
            ~behavioral_mask
        ]

        print(
            f'행동 데이터 고유 ASIN 수 | '
            f'Unique behavioral ASINs : '
            f'{len(behavioral_asins):,}'
        )

        print(
            f'완전 콜드 스타트 ASIN 수 | '
            f'Completely cold-start ASINs : '
            f'{len(cold_asins):,}'
        )

        print(
            f'행동 데이터 카탈로그 커버리지 | '
            f'Behavioral catalog coverage : '
            f'{len(behavioral_asins) / len(catalog_asins) * 100:.2f}%'
        )

        print('---' * 25)

        pbar.update(1)

        pbar.set_description('4/6 Splitting product catalog')

        catalog_behavior_mask = (
            catalog['id'].isin(behavioral_set)
        )

        actual_catalog = (
            catalog.loc[catalog_behavior_mask]
            .copy()
        )

        cold_catalog = (
            catalog.loc[~catalog_behavior_mask]
            .copy()
        )

        items_dict = dict(
            zip(
                behavioral_asins,
                range(1, len(behavioral_asins) + 1)
            )
        )

        actual_catalog['num_id'] = (
            actual_catalog['id']
            .map(items_dict)
            .astype('int32')
        )

        print(
            f'임베딩 어휘 크기 | '
            f'Behavioral vocabulary size : '
            f'{len(items_dict):,}'
        )

        print(
            '패딩 인덱스 | PAD index : 0'
        )

        print('---' * 25)

        del behavioral_set
        del behavioral_index

        pbar.update(1)

        pbar.set_description('5/6 Mapping history item IDs')

        for row in tqdm(
            parsed_lists,
            desc='Mapping history IDs',
            leave=False
        ):
            for i, asin in enumerate(row):
                row[i] = items_dict[asin]

        sessions_hist.drop(
            columns=['prev_items'],
            inplace=True
        )

        sessions_hist['prev_items_list'] = (
            parsed_lists
        )

        print(
            '히스토리 상품 ID 변환 완료 | '
            'History item ID mapping completed'
        )

        print('---' * 25)

        pbar.update(1)

        pbar.set_description('6/6 Mapping targets & validating')

        mapped_targets = (
            sessions_hist['next_item']
            .map(items_dict)
        )

        if mapped_targets.isna().any():
            missing_targets = mapped_targets.isna().sum()

            raise ValueError(
                f'알 수 없는 타겟 상품 발견 | '
                f'Unknown target items detected: '
                f'{missing_targets:,}'
            )

        sessions_hist['next_item'] = (
            mapped_targets.astype('int32')
        )

        core_columns = [
            'prev_items_list',
            'next_item',
            'locale'
        ]

        other_columns = [
            column
            for column in sessions_hist.columns
            if column not in core_columns
        ]

        sessions_hist = sessions_hist[
            core_columns + other_columns
        ]

        if sessions_hist['next_item'].isna().any():
            raise ValueError(
                '타겟 변환 실패 | '
                'Target mapping contains NaN'
            )

        if not sessions_hist[
            'prev_items_list'
        ].map(bool).all():
            raise ValueError(
                '빈 히스토리 발견 | '
                'Empty history detected after mapping'
            )

        if actual_catalog['num_id'].isna().any():
            raise ValueError(
                '상품 num_id 생성 실패 | '
                'Missing product num_id detected'
            )

        if len(items_dict) != len(behavioral_asins):
            raise ValueError(
                '상품 사전 크기 불일치 | '
                'Item dictionary size mismatch'
            )

        target_min = sessions_hist[
            'next_item'
        ].min()

        target_max = sessions_hist[
            'next_item'
        ].max()

        if target_min < 1:
            raise ValueError(
                '잘못된 타겟 ID 발견 | '
                'Invalid target ID below 1'
            )

        if target_max > len(items_dict):
            raise ValueError(
                '잘못된 타겟 ID 발견 | '
                'Target ID exceeds vocabulary size'
            )

        pbar.update(1)

    print()
    print('=' * 90)
    print('최종 처리 결과 | FINAL PROCESSING RESULTS')
    print('=' * 90)

    print(
        f'전체 고유 ASIN 수 | '
        f'Total unique catalog ASINs : '
        f'{len(catalog_asins):,}'
    )

    print(
        f'학습에 사용되는 ASIN 수 | '
        f'Behavioral ASINs : '
        f'{len(behavioral_asins):,}'
    )

    print(
        f'콜드 스타트 ASIN 수 | '
        f'Cold-start ASINs : '
        f'{len(cold_asins):,}'
    )

    print(
        f'처리된 세션 수 | '
        f'Processed sessions : '
        f'{len(sessions_hist):,}'
    )

    print(
        f'상품 임베딩 인덱스 범위 | '
        f'Item embedding ID range : '
        f'1 ... {len(items_dict):,}'
    )

    print(
        '패딩 인덱스 | PAD index : 0'
    )

    print('---' * 25)
    print(
        '파이프라인이 성공적으로 실행되었습니다! | '
        'PIPELINE EXECUTED SUCCESSFULLY!'
    )

    return (
        sessions_hist,
        actual_catalog,
        cold_catalog,
        items_dict
    )

In [ ]:
sessions2, products2, cold_products, items_dict = parse_sessions(sessions1, products1)

In [ ]:
def pre_final_preprocessing(products, sessions): 
    sessions_hist = sessions.copy()
    sessions_hist = sessions_hist.sample(frac = 1, random_state = 42).reset_index(drop = True)
    catalog = products.copy()
    
    print(f'데이터 분할 및 처리 | DATA SPLITTING AND PROCESSING')
    print(f'---' * 40)
    
    print(f'세션 정렬 및 분할 | SESSION SORTING AND SPLITTING')
    print(f'---' * 40)
    
    sessions_hist['session_len'] = sessions_hist['prev_items_list'].str.len()
    sessions_hist['items_list_str'] = sessions_hist['prev_items_list'].astype('str')
    sessions_dict = dict(Counter(sessions_hist['items_list_str']))
    sessions_hist['num_repeat'] = sessions_hist['items_list_str'].map(sessions_dict)
    
    repeat_more_than_1 = sessions_hist[sessions_hist['num_repeat'] > 1]
    repeat_once = sessions_hist[sessions_hist['num_repeat'] == 1]
    print(f"중복 세션 수 | Number of duplicate sessions : {len(repeat_more_than_1)}")
    print(f"고유 세션 수 | Number of unique sessions : {len(repeat_once)}")
    
    train_len_threshold = round(len(sessions_hist) * 0.8)
    more_to_take = train_len_threshold - len(repeat_more_than_1)
    
    sessions_train = pd.concat([repeat_more_than_1, repeat_once.iloc[0: more_to_take]],
                               ignore_index = True)
    
    val_test = repeat_once.iloc[more_to_take:]
    val = val_test.iloc[0 : (len(val_test) // 2)]
    test = val_test.iloc[len(val_test) // 2 :]
    assert len(sessions_hist) == len(sessions_train) + len(val) + len(test)
    
    sessions_preprocessor = ColumnTransformer(
        transformers = [
            ('ohe_locale', OneHotEncoder(sparse_output = False,
                                         handle_unknown = 'ignore'), 
             ['locale'])
        ], 
        remainder = 'passthrough'
    )
    
    sessions_preprocessor.set_output(transform="pandas")
    sessions_train = sessions_preprocessor.fit_transform(sessions_train)
    val = sessions_preprocessor.transform(val)
    test = sessions_preprocessor.transform(test)
    
    sessions_train = sessions_train.drop(columns = 
                                         ['remainder__session_len','remainder__items_list_str', 'remainder__num_repeat'])
    val = val.drop(columns = 
                   ['remainder__session_len','remainder__items_list_str', 'remainder__num_repeat'])
    test = test.drop(columns = 
                     ['remainder__session_len','remainder__items_list_str', 'remainder__num_repeat'])
    
    print(f'세션 데이터셋 생성 완료! | SESSION DATASETS CREATED SUCCESSFULLY!')
    print(f'---' * 40)
    print(f'학습 데이터셋 크기 | Training dataset size : {len(sessions_train)}')
    print(f'검증 데이터셋 크기 | Validation dataset size : {len(val)}')
    print(f'테스트 데이터셋 크기 | Test dataset size {len(test)}')
    print(f'---' * 40)
    print(f'상품 카탈로그 처리 | ITEM CATALOG PROCESSING')
    print(f'---' * 40)
    
    for one_locale in catalog['locale'].unique():
        products_with_outlier_price = catalog[(catalog['locale'] == one_locale) & (catalog['price'] == catalog['price'].max())].index
        region_median = catalog[catalog['locale'] == one_locale]['price'].median()
        catalog.loc[products_with_outlier_price, 'price'] = region_median
        assert (catalog.loc[products_with_outlier_price, 'price'] == region_median).sum() == len(products_with_outlier_price)
    
    print(f'이상치 처리 완료! | OUTLIERS PROCESSED')
    print(f'---' * 40)
    print(f'가격 표준화 | PRICE STANDARDIZATION')
    
    for one_locale in catalog['locale'].unique():
        region_indexes = catalog[catalog['locale'] == one_locale].index
        region_scaler = StandardScaler(
            
        )
        region_prices = np.array(catalog.loc[region_indexes, 
                                    'price']).reshape(-1,1)
        catalog.loc[region_indexes, 
                    'price'] = region_scaler.fit_transform(
            region_prices
        ).flatten()

        assert np.isclose(catalog[catalog['locale'] == one_locale]['price'].mean(), 0 , atol = 1e-7)
        assert np.isclose(catalog[catalog['locale'] == one_locale]['price'].std(ddof = 0), 1 , atol = 1e-7)    
    
    print(f'표준화 완료 | STANDARDIZATION SUCCESSFUL')
    print(f'---' * 40)
    print(f'지역 인코딩 | REGION ENCODING')
    
    catalog_OHE = OneHotEncoder( 
                                handle_unknown = 'ignore',
                                sparse_output = False)
    
    catalog_preprocessor = ColumnTransformer(
        transformers = [
            ('Catalog_OHE', 
            catalog_OHE, 
            ['locale'])
            
        ], 
        remainder = 'passthrough', 
        sparse_threshold = 0
    )
    
    catalog_preprocessor.set_output(transform = 'pandas')
    final_catalog = catalog_preprocessor.fit_transform(catalog)
    assert len(final_catalog.columns) == len(catalog.columns) + len(catalog['locale'].unique()) - 1
    print(f'파이프라인 실행 완료! | PIPELINE EXECUTED SUCCESSFULLY!')
    
    return final_catalog, test, val, sessions_train
    
    
        

    

In [ ]:
catalog, sessions_test, sessions_val, sessions_train = pre_final_preprocessing(products2, sessions2)

In [ ]:
def final_preprocessing(catalog, train, val, test):
    print(f'학습용 데이터셋 분할 완료 | SPLITTING DATASETS INTO TRAINING-READY SAMPLES')
    print(f'---' * 40)
    catalog = catalog.copy()
    train = train.copy()
    val = val.copy()
    test = test.copy()
    
    num_columns = catalog.select_dtypes(
        include=np.number
    ).columns
    
    num_catalog = catalog[num_columns].copy()
    
    mapping = {
        'remainder__title': 'Title',
        'remainder__brand': 'Brand',
        'remainder__color': 'Color',
        'remainder__size': 'Size',
        'remainder__model': 'Model',
        'remainder__material': 'Material',
        'remainder__author': 'Author',
        'remainder__desc': 'Product Description'
    }

    def build_product_info(row): 
        parts = [
            f'{label} | {row[col]}'
            for col, label in mapping.items()
            if col in row
            and pd.notna(row[col])
            and str(row[col]).strip() != ''
        ]
        
        return ' '.join(parts) if parts else None 
    
    temp_catalog = catalog.copy()
    temp_catalog['product_info'] = temp_catalog.apply(
        build_product_info,
        axis=1
    )
    
    valid_catalog = (
        temp_catalog
        .dropna(subset=['product_info'])
        .reset_index(drop=True)
    )
    
    product_info = valid_catalog['product_info'].tolist()
    
    embeddings_ids = valid_catalog[
        'remainder__num_id'
    ].to_numpy(dtype=np.int32)
    
    locale_columns = [
        col
        for col in valid_catalog.columns
        if col.startswith('Catalog_OHE__locale_')
    ]
    
    embedding_metadata = (
        valid_catalog[
            ['remainder__num_id'] + locale_columns
        ]
        .copy()
        .reset_index(drop=True)
    )
    
    embedding_metadata['embedding_row'] = np.arange(
        len(embedding_metadata),
        dtype=np.int32
    )
    
    num_catalog.drop(
        columns=['remainder__num_id'],
        inplace=True
    )
    
    assert len(product_info) == len(embeddings_ids)
    assert len(product_info) == len(embedding_metadata)
    
    print(f'상품 정보 생성 완료 | ITEM INFORMATION CREATION COMPLETED')
    print(f'---' * 40)
    
    train_X = train.drop(
        columns=['remainder__next_item']
    )
    
    train_y = train['remainder__next_item']
    
    val_X = val.drop(
        columns=['remainder__next_item']
    )
    
    val_y = val['remainder__next_item']
    
    test_X = test.drop(
        columns=['remainder__next_item']
    )
    
    test_y = test['remainder__next_item']
    
    assert len(train_X) == len(train_y)
    assert len(val_X) == len(val_y)
    assert len(test_X) == len(test_y)
    
    print(f'샘플링 생성 완료 | SAMPLE CREATION COMPLETED')
    
    return  num_catalog,product_info,embeddings_ids,embedding_metadata,train_X,train_y,val_X,val_y,test_X,test_y


In [ ]:
num_catalog, product_info, embeddings_ids, embedding_metadata, train_X, train_y, val_X, val_y, test_X, test_y = final_preprocessing(catalog,sessions_train,sessions_val,sessions_test)

In [ ]:
MODEL = 'voyage-4-large'
EMBEDDING_DIM = 1024
BATCH_SIZE = 128
DTYPE = np.float16

PROJECT_ROOT = Path.cwd().parent
EMBEDDINGS_DIR = PROJECT_ROOT / 'artifacts' / 'embeddings' / 'voyage4_large_1024'
EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)

EMBEDDINGS_PATH = EMBEDDINGS_DIR / 'product_embeddings.npy'
IDS_PATH = EMBEDDINGS_DIR / 'embedding_num_ids.npy'
METADATA_PATH = EMBEDDINGS_DIR / 'embedding_metadata.pkl'
CONFIG_PATH = EMBEDDINGS_DIR / 'config.json'
PROGRESS_PATH = EMBEDDINGS_DIR / 'progress.txt'

if 'VOYAGE_API_KEY' not in os.environ:
    os.environ['VOYAGE_API_KEY'] = getpass('Voyage API key: ')

vo = voyageai.Client(
    max_retries=10,
    timeout=120
)

N = len(product_info)

assert len(product_info) == len(embeddings_ids)
assert len(product_info) == len(embedding_metadata)
assert embedding_metadata['embedding_row'].to_numpy().tolist() == list(range(N))

np.save(
    IDS_PATH,
    np.asarray(embeddings_ids, dtype=np.int32)
)

metadata_to_save = embedding_metadata.copy()
metadata_to_save.to_pickle(METADATA_PATH)

config = {
    'model': MODEL,
    'embedding_dimension': EMBEDDING_DIM,
    'dtype': 'float16',
    'input_type': None,
    'batch_size': BATCH_SIZE,
    'number_of_representations': N
}

with open(CONFIG_PATH, 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=4)

print(f'모델 | Model: {MODEL}')
print(f'상품 표현 수 | Product representations: {N:,}')
print(f'임베딩 차원 | Embedding dimension: {EMBEDDING_DIM}')
print(f'배치 크기 | Batch size: {BATCH_SIZE}')
print(f'예상 파일 크기 | Expected embedding file size: {N * EMBEDDING_DIM * np.dtype(DTYPE).itemsize / 1024**3:.2f} GB')
print(f'저장 경로 | Save directory: {EMBEDDINGS_DIR}')
print('---' * 35)

test_n = min(10, N)

test_response = vo.embed(
    texts=product_info[:test_n],
    model=MODEL,
    input_type=None,
    output_dimension=EMBEDDING_DIM
)

test_embeddings = np.asarray(
    test_response.embeddings,
    dtype=np.float32
)

assert test_embeddings.shape == (test_n, EMBEDDING_DIM)
assert np.isfinite(test_embeddings).all()

print(f'API 연결 테스트 성공 | API connection test successful')
print(f'테스트 임베딩 크기 | Test embedding shape: {test_embeddings.shape}')
print('---' * 35)

if EMBEDDINGS_PATH.exists():
    embeddings = np.load(
        EMBEDDINGS_PATH,
        mmap_mode='r+'
    )

    if embeddings.shape != (N, EMBEDDING_DIM):
        raise ValueError(
            f'Existing embedding file has wrong shape: {embeddings.shape}, '
            f'expected {(N, EMBEDDING_DIM)}'
        )

    if embeddings.dtype != DTYPE:
        raise ValueError(
            f'Existing embedding file has wrong dtype: {embeddings.dtype}, '
            f'expected {DTYPE}'
        )
else:
    embeddings = np.lib.format.open_memmap(
        EMBEDDINGS_PATH,
        mode='w+',
        dtype=DTYPE,
        shape=(N, EMBEDDING_DIM)
    )

if PROGRESS_PATH.exists():
    start_position = int(PROGRESS_PATH.read_text(encoding='utf-8').strip())
else:
    start_position = 0

if start_position < 0 or start_position > N:
    raise ValueError(
        f'Invalid saved progress: {start_position}'
    )

print(f'시작 행 | Starting embedding row: {start_position:,} / {N:,}')
print('---' * 35)

if start_position < N:
    total_batches = (N + BATCH_SIZE - 1) // BATCH_SIZE
    initial_batch = start_position // BATCH_SIZE

    with tqdm(
        total=total_batches,
        initial=initial_batch,
        desc='Creating product embeddings'
    ) as pbar:

        for start in range(
            start_position,
            N,
            BATCH_SIZE
        ):
            end = min(
                start + BATCH_SIZE,
                N
            )

            response = vo.embed(
                texts=product_info[start:end],
                model=MODEL,
                input_type=None,
                output_dimension=EMBEDDING_DIM
            )

            batch_embeddings = np.asarray(
                response.embeddings,
                dtype=DTYPE
            )

            expected_shape = (
                end - start,
                EMBEDDING_DIM
            )

            if batch_embeddings.shape != expected_shape:
                raise ValueError(
                    f'Wrong embedding shape at rows {start}:{end}: '
                    f'{batch_embeddings.shape}, expected {expected_shape}'
                )

            if not np.isfinite(batch_embeddings).all():
                raise ValueError(
                    f'NaN or Inf detected at rows {start}:{end}'
                )

            embeddings[start:end] = batch_embeddings
            embeddings.flush()

            PROGRESS_PATH.write_text(
                str(end),
                encoding='utf-8'
            )

            pbar.update(1)

embeddings.flush()

saved_embeddings = np.load(
    EMBEDDINGS_PATH,
    mmap_mode='r'
)

saved_ids = np.load(
    IDS_PATH
)

saved_metadata = pd.read_pickle(
    METADATA_PATH
)

assert saved_embeddings.shape == (N, EMBEDDING_DIM)
assert saved_ids.shape[0] == N
assert len(saved_metadata) == N
assert np.isfinite(saved_embeddings[:min(1000, N)]).all()
assert np.isfinite(saved_embeddings[max(0, N - 1000):]).all()
assert not np.all(saved_embeddings[0] == 0)
assert not np.all(saved_embeddings[-1] == 0)

PROGRESS_PATH.write_text(
    str(N),
    encoding='utf-8'
)

print()
print('=' * 100)
print('임베딩 생성 완료! | EMBEDDING GENERATION COMPLETED!')
print('=' * 100)
print(f'임베딩 행렬 | Embedding matrix: {saved_embeddings.shape}')
print(f'데이터 타입 | Dtype: {saved_embeddings.dtype}')
print(f'상품 ID 수 | Number of mapped IDs: {len(saved_ids):,}')
print(f'메타데이터 행 수 | Metadata rows: {len(saved_metadata):,}')
print(f'임베딩 파일 | Embeddings: {EMBEDDINGS_PATH}')
print(f'ID 파일 | IDs: {IDS_PATH}')
print(f'메타데이터 파일 | Metadata: {METADATA_PATH}')
print(f'설정 파일 | Config: {CONFIG_PATH}')
print('=' * 100)

In [ ]:
PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
MAPPINGS_DIR = PROJECT_ROOT / 'artifacts' / 'mappings'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MAPPINGS_DIR.mkdir(parents=True, exist_ok=True)

train_processed = train_X.copy()
train_processed['target'] = train_y.to_numpy()

val_processed = val_X.copy()
val_processed['target'] = val_y.to_numpy()

test_processed = test_X.copy()
test_processed['target'] = test_y.to_numpy()

train_processed.to_parquet(
    PROCESSED_DIR / 'train.parquet',
    index=False
)

val_processed.to_parquet(
    PROCESSED_DIR / 'val.parquet',
    index=False
)

test_processed.to_parquet(
    PROCESSED_DIR / 'test.parquet',
    index=False
)

In [ ]:
num_catalog_to_save = num_catalog.copy()

num_catalog_to_save.insert(
    0,
    'num_id',
    catalog['remainder__num_id'].to_numpy()
)

num_catalog_to_save.to_parquet(
    PROCESSED_DIR / 'catalog_numeric.parquet',
    index=False
)

In [ ]:
with open(
    MAPPINGS_DIR / 'items_dict.pkl',
    'wb'
) as f:
    pickle.dump(items_dict, f)